# CaV1.2 WT | vanilla versus corrected masked WT

This focused notebook compares the unchanged vanilla WT ensemble with the
gap-aware corrected WT masked candidate (`masked_v3new`). All plotted
coordinates come from the external reconciliation data root and use the
registered 3 Å recycle-convergence selection. Corrected mutant masked
cohorts are intentionally not substituted or analyzed here.

The experimental markers are local-geometry references. They do not by
themselves establish a functional state or replace structural QC.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv, cav12_active_masked_distance_path
from shared.plotting import CAV12_PALETTE, plot_protocol_split_with_experimentals
sns.set_context('notebook')
selection = 'all_ok_3'
output_root = repo_root / 'analysis' / 'cav12_wt_v3new_comparison'
figure_root = output_root / 'figures'
table_root = output_root / 'tables'
figure_root.mkdir(parents=True, exist_ok=True)
table_root.mkdir(parents=True, exist_ok=True)


In [ ]:
data = repo_root / 'cav12' / 'dataDistances'
vanilla_options = distance_csv_options(repo_root, data / '26-02-10_Cav12_wt_vanillaAF2_distances_all.csv', 'Cav12', 'WT', 'vanilla')
masked_relpath = cav12_active_masked_distance_path(repo_root, 'WT')
masked_options = distance_csv_options(repo_root, masked_relpath, 'Cav12', 'WT', 'masked')
vanilla = load_selected_distance_csv('CaV1.2 WT vanilla', vanilla_options, selection, fallback_to_all=False)
masked = load_selected_distance_csv('CaV1.2 WT corrected masked', masked_options, selection, fallback_to_all=False)
print('Registered masked distance path:', masked_relpath)
print('Selection:', selection)
print('Rows:', {'vanilla': len(vanilla), 'corrected_masked': len(masked)})
print('Unique model files:', {'vanilla': vanilla.pdb_file.nunique(), 'corrected_masked': masked.pdb_file.nunique()})
print('Vanilla source:', vanilla_options[selection])
print('Corrected masked source:', masked_options[selection])
assert len(vanilla) > 0 and len(masked) > 0


In [ ]:
FILTER_ALIASES = {
    'E363–E706': 'CA_GLU363_CA-GLU706_CA',
    'E363–E1135': 'CA_GLU363_CA-GLU1135_CA',
    'E363–E1464': 'CA_GLU363_CA-GLU1464_CA',
    'E706–E1135': 'CA_GLU706_CA-GLU1135_CA',
    'E706–E1464': 'CA_GLU706_CA-GLU1464_CA',
    'E1135–E1464': 'CA_GLU1135_CA-GLU1464_CA',
}
GATE_ALIASES = {
    'L401–L749': 'CA_LEU401_CA-LEU749_CA',
    'L401–V1182': 'CA_LEU401_CA-VAL1182_CA',
    'L401–I1516': 'CA_LEU401_CA-ILE1516_CA',
    'L749–V1182': 'CA_LEU749_CA-VAL1182_CA',
    'L749–I1516': 'CA_LEU749_CA-ILE1516_CA',
    'V1182–I1516': 'CA_VAL1182_CA-ILE1516_CA',
}
VSD_ALIASES = {
    'F170–R237': 'CA_ARG237_CA-PHE170_CA',
    'F170–R246': 'CA_ARG246_CA-PHE170_CA',
    'F567–R620': 'CA_ARG620_CA-PHE567_CA',
    'F567–K629': 'CA_LYS629_CA-PHE567_CA',
    'F940–K1015': 'CA_LYS1015_CA-PHE940_CA',
    'F940–K1031': 'CA_ARG1031_CA-PHE940_CA',
    'F1278–R1377': 'CA_ARG1377_CA-PHE1278_CA',
    'F1278–K1386': 'CA_LYS1386_CA-PHE1278_CA',
}
EXPERIMENTAL = [
    ('8HLP', {'E363–E706': 6.7, 'E363–E1135': 9.4, 'E363–E1464': 6.8, 'E706–E1135': 5.9, 'E706–E1464': 9.0, 'E1135–E1464': 6.7, 'L401–L749': 8.9, 'L401–V1182': 9.3, 'L401–I1516': 10.4, 'L749–V1182': 8.1, 'L749–I1516': 14.1, 'V1182–I1516': 7.1}),
    ('8WE6', {'E363–E706': 7.2, 'E363–E1135': 10.4, 'E363–E1464': 7.2, 'E706–E1135': 6.4, 'E706–E1464': 9.4, 'E1135–E1464': 7.3, 'L401–L749': 9.4, 'L401–V1182': 9.9, 'L401–I1516': 10.2, 'L749–V1182': 8.2, 'L749–I1516': 13.9, 'V1182–I1516': 7.2}),
    ('8FD7', {'E363–E706': 7.6, 'E363–E1135': 10.6, 'E363–E1464': 7.4, 'E706–E1135': 6.5, 'E706–E1464': 12.3, 'E1135–E1464': 10.9, 'L401–L749': 11.1, 'L401–V1182': 12.1, 'L401–I1516': 11.8, 'L749–V1182': 8.3, 'L749–I1516': 16.0, 'V1182–I1516': 10.7}),
]
exp_labels = [f'Experimental | {name}' for name, _ in EXPERIMENTAL]
exp_by_alias = lambda aliases: [{alias: [values[alias]] for alias in aliases if alias in values} for _, values in EXPERIMENTAL]
exp_colors = ['#E57373', '#F48FB1', '#FFB74D']


In [ ]:
def plot_region(name, aliases, filename, title, colors):
    ax = plot_protocol_split_with_experimentals(
        vanilla, masked, aliases, title, colors,
        exp_distances_list=exp_by_alias(aliases),
        dataset_labels=exp_labels, pdb_colors=exp_colors,
        fig_width=12, fig_height=6.2,
    )
    ax.figure.savefig(figure_root / filename, dpi=300, bbox_inches='tight')
    return ax

plot_region('selectivity_filter', FILTER_ALIASES, '01_selectivity_filter_vanilla_vs_corrected_masked.png', 'CaV1.2 WT | selectivity-filter Cα distances | vanilla versus corrected masked', ['#B9DCC6', '#4F8060'])
plot_region('intracellular_gate', GATE_ALIASES, '02_intracellular_gate_vanilla_vs_corrected_masked.png', 'CaV1.2 WT | intracellular-gate Cα distances | vanilla versus corrected masked', ['#C9DDF4', '#3978B5'])
plot_region('voltage_sensor', VSD_ALIASES, '03_voltage_sensor_vanilla_vs_corrected_masked.png', 'CaV1.2 WT | voltage-sensor Cα distances | vanilla versus corrected masked', ['#D7E7F2', '#2F7892'])


In [ ]:
def summarize(frame, protocol, aliases):
    rows = []
    for alias, column in aliases.items():
        values = pd.to_numeric(frame[column], errors='coerce').dropna()
        rows.append({'protocol': protocol, 'region': name, 'distance_alias': alias, 'n': len(values), 'median_A': values.median(), 'q25_A': values.quantile(.25), 'q75_A': values.quantile(.75), 'min_A': values.min(), 'max_A': values.max()})
    return rows
summary_rows = []
for name, aliases in [('selectivity_filter', FILTER_ALIASES), ('intracellular_gate', GATE_ALIASES), ('voltage_sensor', VSD_ALIASES)]:
    for protocol, frame in [('vanilla', vanilla), ('corrected_masked', masked)]:
        for alias, column in aliases.items():
            values = pd.to_numeric(frame[column], errors='coerce').dropna()
            summary_rows.append({'protocol': protocol, 'region': name, 'distance_alias': alias, 'n': len(values), 'median_A': values.median(), 'q25_A': values.quantile(.25), 'q75_A': values.quantile(.75), 'min_A': values.min(), 'max_A': values.max()})
summary = pd.DataFrame(summary_rows)
summary.to_csv(table_root / 'wt_vanilla_vs_masked_v3new_distance_summary.tsv', sep='\t', index=False)
display(summary.round(2))


## Interpretation guardrails

These panels answer whether the corrected masked WT ensemble broadens or
redistributes local geometry relative to unchanged vanilla WT and whether
experimental reference distances are represented. They do not certify
clashscore, pLDDT, interface quality, or global experimental-reference RMSD;
those remain separate QC inputs.